# Finetune VisDrone — **một model, một tab**

Notebook này train **đúng một model**. Mở 4 tab Colab, mỗi tab chạy notebook
này, **chỉ đổi một dòng duy nhất** ở cell 1:

```python
MODEL = "v8n-base"     # tab 1
MODEL = "v11n-base"    # tab 2
MODEL = "v26n-base"    # tab 3
MODEL = "v26n-p2"      # tab 4
```

Toàn bộ phần còn lại giữ nguyên, không sửa gì. Mỗi tab một GPU, mỗi tab lưu
vào thư mục Drive riêng nên không đè lên nhau. Cell cuối gom kết quả từ mọi
model đã xong thành một bảng.

> **Colab giới hạn số session chạy cùng lúc** (Pro thường 2–3). Mở 4 tab mà bị
> từ chối thì chạy 2 tab trước, xong rồi chạy 2 tab sau — kết quả gom vẫn đủ.

---

## Bốn điều đã kiểm chứng trên `ultralytics==8.4.118`

Không phải suy đoán; mỗi dòng dưới đây đều đo được, và mỗi dòng đều đổi một
chỗ trong notebook.

### 1. `yolo26-p2.yaml` có sẵn trong package — không cần tải

File trong `ultralytics==8.4.118` khớp đúng bản trên GitHub main:
`end2end: True`, `reg_max: 1`, `Detect(P2, P3, P4, P5)` tại `[19, 22, 25, 28]`,
scale `n` → **2.662.400 tham số** (dựng thử ra đúng 2.66M).

### 2. YOLO26 **không dùng NMS** — output khác hẳn

| Model | Output thô | NMS |
|---|---|---|
| `yolov8n`, `yolo11n` | `(1, 4+nc, 8400)` | cần |
| **`yolo26n`, `yolo26n-p2`** | **`(1, 300, 6)`** | **không** |

`(1, 300, 6)` là box decode sẵn `[x1, y1, x2, y2, conf, cls]`.

**Lợi:** pipeline hiện tại tốn **18 ms/frame** NMS trên CPU, so với 47 ms
inference — bỏ được là khoản cắt lớn nhất còn lại.
**Phải xử lý:** cắm v26 vào `3-pipeline/detector.py` sẽ **sai thầm lặng**,
không lỗi. Cell export ghi rõ shape từng model.

### 3. Đầu end-to-end cắt cứng ở **300** detection

Giao thức VisDrone chấm ở `maxDets=500`, mà val của bạn trung bình **345
det/ảnh** ở conf 0.001. Để mặc định thì v26 bị thiệt mà không có dấu hiệu gì.
Notebook nâng `max_det = 500`, đã verify output đổi thành `(1, 500, 6)`.

### 4. `yolo26n-p2.pt` **không tồn tại** — chỉ 40% trọng số nạp được

Phải dựng từ yaml rồi nạp một phần từ `yolo26n.pt`. Đo thật, bằng cách tải
pretrained về rồi đếm tensor trùng tên **và** trùng shape:

| Model | Trọng số nạp được |
|---|---|
| `v8n-base` | 355/355 — **100%** |
| `v11n-base` | 499/499 — **100%** |
| `v26n-base` | 708/708 — **100%** |
| **`v26n-p2`** | **360/902 — 40%** |

Tức **60% model p2 khởi tạo ngẫu nhiên**, trong khi ba model kia nạp đủ. Nên
p2 mặc định được **1.5× epoch**, và tỉ lệ này được ghi vào `summary.json` rồi
lên bảng so sánh. Thiếu con số đó, bảng sẽ bị đọc thành "kiến trúc p2 kém hơn",
trong khi thực ra nó chỉ xuất phát sau.

Thêm: p2 có stride `[4, 8, 16, 32]` thay vì `[8, 16, 32]` → **34.000 anchor**
thay vì 8.400 ở 640px (đo được). Đó là lý do batch mặc định của p2 thấp hơn —
và cũng là lý do nó đáng thử với VisDrone, nơi vật thể rất nhỏ.

## 1. Chọn model — **dòng duy nhất cần sửa giữa các tab**

In [ ]:
# ============================================================
#  DOI DUNG DONG NAY O MOI TAB. Khong sua gi khac.
# ============================================================
MODEL = "v8n-base"        # "v8n-base" | "v11n-base" | "v26n-base" | "v26n-p2"
# ============================================================

SEED = 0
IMGSZ = 640
EPOCHS_BASE = 50

# Early stopping: dung khi mAP50-95 khong cai thien sau PATIENCE epoch lien
# tiep. Ultralytics tu lam, chi can truyen patience -- va no luu lai best.pt
# cua epoch tot nhat chu khong phai epoch cuoi, nen dung som khong mat gi.
#
# 15 chu khong phai 30: patience phai nho hon nhieu so voi tong so epoch, neu
# khong thi no khong bao gio kip kich hoat. Voi 50 epoch thi patience 30 nghia
# la phai te lien tuc tu epoch 20 tro di moi dung -- gan nhu khong xay ra, tuc
# la co early stopping tren giay ma thuc te khong bao gio chay.
PATIENCE = 15

# maxDets cua giao thuc VisDrone. Dau end2end mac dinh cat o 300, ma anh
# VisDrone dong co the vuot 300 vat the -> khong nang len la v26 bi thiet
# ma khong bao gi.
MAX_DET = 500

REGISTRY = {
    "v8n-base":  dict(cfg="yolov8n.yaml",    weights="yolov8n.pt",
                      batch=128, epochs=EPOCHS_BASE),
    "v11n-base": dict(cfg="yolo11n.yaml",    weights="yolo11n.pt",
                      batch=128, epochs=EPOCHS_BASE),
    "v26n-base": dict(cfg="yolo26n.yaml",    weights="yolo26n.pt",
                      batch=128, epochs=EPOCHS_BASE),
    # Khong co yolo26n-p2.pt -> dung tu yaml, nap mot phan tu yolo26n.pt.
    # Do that: chi 360/902 tensor nap duoc (40%), tuc 60% model khoi tao ngau
    # nhien, trong khi 3 model kia nap du 100%. -> can nhieu epoch hon.
    # Them tang P2 (stride 4) -> 34000 anchor thay vi 8400 -> batch thap hon.
    "v26n-p2":   dict(cfg="yolo26n-p2.yaml", weights="yolo26n.pt",
                      batch=64,  epochs=int(EPOCHS_BASE * 1.5)),
}

assert MODEL in REGISTRY, f"MODEL phai la mot trong {list(REGISTRY)}"
SPEC = dict(REGISTRY[MODEL])

print(f"Tab nay train : {MODEL}")
print(f"  cfg      : {SPEC['cfg']}")
print(f"  weights  : {SPEC['weights']}")
print(f"  batch    : {SPEC['batch']} (cell 9 se do lai va tu chinh)")
print(f"  epochs   : {SPEC['epochs']} (toi da)")
print(f"  patience : {PATIENCE} -> dung som neu {PATIENCE} epoch lien tiep khong cai thien")
print(f"  max_det  : {MAX_DET}")

## 2. Kiểm tra GPU

In [ ]:
import subprocess

print(subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True).stdout or "!! khong thay nvidia-smi")

import torch
if torch.cuda.device_count() == 0:
    raise SystemExit("Khong co GPU. Runtime > Change runtime type > GPU (A100).")

GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nGPU  : {GPU_NAME}")
print(f"VRAM : {VRAM_GB:.1f} GB")

if "A100" not in GPU_NAME:
    print(f"\n[canh bao] Khong phai A100. Batch mac dinh tinh cho A100 40GB; "
          f"cell 9 se do lai va ha xuong cho vua {VRAM_GB:.0f} GB.")

## 3. Cài đặt

Ghim đúng version đã kiểm chứng. `yolo26` chỉ có từ ultralytics 8.4.x — bản cũ
hơn báo "model not found" cho hai model v26.

In [ ]:
%pip install -q "ultralytics==8.4.118" onnx onnxslim onnxruntime

import ultralytics, torch, platform
print("ultralytics", ultralytics.__version__)
print("torch      ", torch.__version__, "| cuda", torch.version.cuda)
print("python     ", platform.python_version())

from ultralytics.utils.downloads import GITHUB_ASSETS_NAMES
w = SPEC["weights"]
print(f"\n{w}: {'co pretrained' if w in GITHUB_ASSETS_NAMES else 'KHONG co'}")
if MODEL == "v26n-p2":
    print("  (dung: yolo26n-p2.pt khong ton tai. Dung yolo26n.pt nap mot phan.)")

## 4. Mount Drive

**Trước khi chạy**: mở link dataset → **Add shortcut to Drive → My Drive**.
Thư mục chia sẻ không tự xuất hiện trong `MyDrive` nếu chưa tạo shortcut, và
`gdown --folder` bị chặn ở 50 file nên vô dụng với dataset vài nghìn ảnh.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
ROOT = "/content/drive/MyDrive"
print("Thu muc cap 1 trong MyDrive:\n")
for d in sorted(os.listdir(ROOT))[:60]:
    if os.path.isdir(os.path.join(ROOT, d)):
        print("  ", d)

## 5. Lấy dataset — chọn **một trong hai cách**

Dataset nằm ở **"Được chia sẻ với tôi"**, và Colab không mount được mục đó.
Không phải vì Colab thiếu tính năng: **"Được chia sẻ với tôi" không phải một
thư mục**, nó là *bộ lọc hiển thị* — không có đường dẫn thật để mount. Colab
chỉ mount `MyDrive` và `Shared drives`.

### Cách A — tải thẳng bằng link (khuyến nghị, **không đụng Drive của bạn**)

Dán link chia sẻ của 2 file zip vào `ZIP_LINKS`. Tải thẳng từ máy chủ Google
về Colab, **không qua FUSE nên nhanh hơn hẳn** cách B, và không thêm gì vào
Drive của bạn.

Lấy link: chuột phải từng file `.zip` → **Chia sẻ** → **Sao chép đường liên
kết**. Cần quyền là **"Bất kỳ ai có đường liên kết"** — nếu đang để "Bị hạn
chế" thì nhờ `votinh42069` đổi, hoặc dùng cách B.

### Cách B — tạo lối tắt

> Chuột phải `visdrone-4` → **Sắp xếp** → **Thêm lối tắt vào Drive**
> → **Drive của tôi**

Lối tắt **không phải bản sao**: chủ sở hữu vẫn là người kia, không tốn dung
lượng Drive của bạn. Nó chỉ tạo một mục thật trong cây My Drive trỏ tới file
đó, để mount có đường dẫn mà đi tới. Để `ZIP_LINKS` trống thì cell tự dò.

In [ ]:
import os, sys, glob, subprocess

# ---- CACH A: dan link chia se vao day (khong dung toi Drive cua ban) ----
ZIP_LINKS = {
    "train": "",   # link chia se cua VisDrone2019-MOT-train.zip
    "val":   "",   # link chia se cua VisDrone2019-MOT-val.zip
}

# ---- CACH B: de trong ZIP_LINKS, cell se tu do trong MyDrive -----------
DATASET_DIR = ""   # dien tay neu muon chi dinh chinh xac

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".JPG", ".JPEG", ".PNG")
ZIP_TRAIN = ZIP_VAL = None

if all(ZIP_LINKS.values()):
    # subprocess chu khong phai %pip: magic nam trong khoi if la thu de vo,
    # va cach nay chay giong nhau du notebook duoc chay bang gi.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown>=5.1"],
                   check=True)
    import gdown
    os.makedirs("/content/zips", exist_ok=True)
    for split, url in ZIP_LINKS.items():
        out = f"/content/zips/VisDrone-MOT-{split}.zip"
        if os.path.exists(out) and os.path.getsize(out) > 1e8:
            print(f"[bo qua] {split}: da tai roi ({os.path.getsize(out)/1e9:.2f} GB)")
        else:
            print(f"Dang tai {split} ...", flush=True)
            # fuzzy: chap nhan link dang /file/d/<id>/view chu khong can tach id
            gdown.download(url=url, output=out, quiet=False, fuzzy=True)
        assert os.path.exists(out), (
            f"Tai {split} that bai. Thuong la do quyen chia se dang 'Bi han che' "
            f"-- doi sang 'Bat ky ai co duong lien ket', hoac dung cach B.")
    ZIP_TRAIN, ZIP_VAL = "/content/zips/VisDrone-MOT-train.zip", \
                         "/content/zips/VisDrone-MOT-val.zip"
    print("\n[cach A] tai truc tiep, khong dung toi Drive cua ban")

else:
    if not DATASET_DIR:
        print("Dang tim file VisDrone .zip trong MyDrive ...")
        hits = set()
        for depth in ("*", "*/*", "*/*/*"):
            for z in glob.glob(f"/content/drive/MyDrive/{depth}.zip"):
                if "visdrone" in os.path.basename(z).lower():
                    hits.add(os.path.dirname(z))
        hits = sorted(hits)
        if len(hits) == 1:
            DATASET_DIR = hits[0]
            print(f"  tim thay: {DATASET_DIR}")
        elif len(hits) > 1:
            for h in hits:
                print("   ", h)
            raise SystemExit("Nhieu noi khop -- dien mot cai vao DATASET_DIR.")

    assert DATASET_DIR and os.path.isdir(DATASET_DIR), (
        "Khong thay dataset trong MyDrive.\n"
        "'Duoc chia se voi toi' KHONG phai thu muc that nen Colab khong mount duoc.\n"
        "  -> Cach A: dan link chia se vao ZIP_LINKS o dau cell nay, hoac\n"
        "  -> Cach B: chuot phai thu muc -> Sap xep -> Them loi tat vao Drive\n"
        "             (loi tat KHONG phai ban sao, khong ton dung luong cua ban)")

    ZIPS = sorted(glob.glob(os.path.join(DATASET_DIR, "*.zip")))
    ZIP_TRAIN = next((z for z in ZIPS if "train" in os.path.basename(z).lower()), None)
    ZIP_VAL = next((z for z in ZIPS if "val" in os.path.basename(z).lower()), None)
    print(f"\n[cach B] doc qua loi tat trong MyDrive: {DATASET_DIR}")

assert ZIP_TRAIN and ZIP_VAL, f"Thieu zip: train={ZIP_TRAIN} val={ZIP_VAL}"
for z in (ZIP_TRAIN, ZIP_VAL):
    print(f"  {os.path.basename(z):34s} {os.path.getsize(z)/1e9:5.2f} GB")

IS_MOT = "MOT" in os.path.basename(ZIP_TRAIN).upper()
print(f"\nDinh dang : {'VisDrone-MOT (theo vet)' if IS_MOT else 'VisDrone-DET / YOLO'}")
if IS_MOT:
    print("  -> anh trong sequences/<ten_seq>/, nhan la 1 file cho ca sequence")
    print("  -> cell 7 se chuyen sang dinh dang YOLO truoc khi train")

## 6. Giải nén về đĩa local

**Đừng train trực tiếp trên Drive.** Drive gắn qua FUSE — mỗi lần mở một file
ảnh là một lượt gọi mạng. Train ở batch 128 cần vài trăm ảnh/giây, FUSE không
đáp ứng nổi; GPU sẽ ngồi chờ và bạn tưởng model chậm trong khi thật ra là I/O.

Chép file `.zip` (một file lớn) rồi giải nén tại chỗ — nhanh hơn hẳn so với
chép hàng chục nghìn ảnh lẻ qua FUSE. Mỗi tab là một runtime riêng nên tab nào
cũng phải làm lại bước này.

In [ ]:
import os, glob, time, shutil, zipfile

LOCAL = "/content/dataset"
os.makedirs(LOCAL, exist_ok=True)

def already_extracted(split):
    """Thu muc giai nen mang ten trong zip (VisDrone2019-...), khong phai ten
    file zip, nen kiem tra theo noi dung chu dung theo ten."""
    for d in glob.glob(os.path.join(LOCAL, "*")):
        if os.path.isdir(d) and split in os.path.basename(d).lower() and (
                os.path.isdir(os.path.join(d, "sequences"))
                or os.path.isdir(os.path.join(d, "images"))):
            return d
    return None


for split, z in (("train", ZIP_TRAIN), ("val", ZIP_VAL)):
    d = already_extracted(split)
    if d:
        print(f"[bo qua] {split} da giai nen: {os.path.basename(d)}")
        continue

    name = os.path.basename(z)
    t0 = time.time()

    # Chi chep khi zip con nam tren Drive (FUSE). Cach A da tai san ve dia
    # local roi -- chep them mot ban 4 GB nua la phi thoi gian va phi dia.
    on_drive = z.startswith("/content/drive")
    if on_drive:
        src = os.path.join("/content", name)
        print(f"Chep {name} ({os.path.getsize(z)/1e9:.2f} GB) tu Drive ...", flush=True)
        shutil.copy(z, src)
        print(f"  chep xong sau {time.time()-t0:.0f}s", flush=True)
    else:
        src = z
        print(f"{name} da o dia local, giai nen thang", flush=True)

    print("  dang giai nen ...", flush=True)
    with zipfile.ZipFile(src) as zf:
        zf.extractall(LOCAL)
    if on_drive:
        os.remove(src)
    print(f"  [ok] {split} sau {(time.time()-t0)/60:.1f} phut")

print("\nDa co trong /content/dataset:")
for d in sorted(os.listdir(LOCAL)):
    print("  ", d)
free = shutil.disk_usage("/content").free / 1e9
print(f"\nDia con trong: {free:.0f} GB")

## 7. Chuyển MOT → YOLO, dựng và **kiểm tra** `data.yaml`

### Vì sao phải chuyển

VisDrone-MOT lưu nhãn **một file cho cả sequence**, mỗi dòng 10 cột:

```
frame_index, target_id, x, y, w, h, score, category, truncation, occlusion
```

YOLO cần **một file cho mỗi ảnh**, 5 cột, toạ độ chuẩn hoá `[0,1]`:

```
class cx cy w h
```

Ba chỗ dễ sai thầm lặng, đều xử lý ở đây:

- **Chỉ số lớp lệch 1.** MOT đánh `1=pedestrian … 10=motor`; YOLO cần `0..9`.
  Quên trừ 1 thì mọi vật thể bị gán sai lớp mà không có lỗi nào báo.
- **`score = 0` là vùng bỏ qua**, không phải vật thể có độ tin cậy thấp. Giữ
  lại là dạy model học rác.
- **`category = 0` (ignored) và `11` (others)** không nằm trong 10 lớp. Giữ lại
  thì `class id >= nc` và train sẽ chết giữa chừng.

### Ảnh liên tiếp gần như trùng nhau

MOT là video ~25–30 fps, nên hai frame cạnh nhau gần như y hệt. Lấy hết vừa
làm epoch chậm gấp mấy lần vừa không thêm bao nhiêu thông tin. `FRAME_STRIDE`
mặc định **3**; đặt `1` nếu bạn muốn dùng mọi frame.

Ảnh được **symlink** chứ không copy — dataset vài GB, nhân đôi là phí đĩa và
phí thời gian.

In [ ]:
import os, glob, yaml, random, collections, shutil, cv2

FRAME_STRIDE = 3      # 1 = lay moi frame

# MOT category -> chi so YOLO. Bo 0 (ignored) va 11 (others).
MOT2YOLO = {1: 0, 2: 1, 3: 2, 4: 3, 5: 4, 6: 5, 7: 6, 8: 7, 9: 8, 10: 9}
VISDRONE_NAMES = ["pedestrian", "people", "bicycle", "car", "van", "truck",
                  "tricycle", "awning-tricycle", "bus", "motor"]

YOLO_ROOT = os.path.join(LOCAL, "yolo")


def _link(src, dst):
    """Symlink de khong nhan doi vai GB anh. Hardlink roi copy la duong lui
    cho he thong file khong cho symlink."""
    if os.path.exists(dst):
        return
    for fn in (os.symlink, os.link, shutil.copy):
        try:
            fn(src, dst)
            return
        except (OSError, NotImplementedError, AttributeError):
            continue
    raise OSError(f"khong tao duoc lien ket lan ban sao cho {src}")


def find_mot_root(local, want):
    for d in glob.glob(os.path.join(local, "*")):
        if (os.path.isdir(os.path.join(d, "sequences"))
                and os.path.isdir(os.path.join(d, "annotations"))
                and want in os.path.basename(d).lower()):
            return d
    return None


def convert_mot(mot_root, split, stride):
    """MOT -> YOLO. Tra ve (so anh, so box, cac van de gap phai)."""
    img_out = os.path.join(YOLO_ROOT, split, "images")
    lab_out = os.path.join(YOLO_ROOT, split, "labels")
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lab_out, exist_ok=True)

    seq_dirs = sorted(glob.glob(os.path.join(mot_root, "sequences", "*")))
    n_img = n_box = n_skip_ignored = n_skip_cat = 0
    issues = []

    for si, seq in enumerate(seq_dirs):
        name = os.path.basename(seq)
        ann = os.path.join(mot_root, "annotations", name + ".txt")
        if not os.path.exists(ann):
            issues.append(f"{name}: khong co file annotation")
            continue

        frames = sorted(f for f in os.listdir(seq) if f.endswith(IMG_EXT))
        if not frames:
            continue
        # Moi frame trong mot sequence cung kich thuoc -> doc mot lan.
        probe = cv2.imread(os.path.join(seq, frames[0]))
        if probe is None:
            issues.append(f"{name}: khong doc duoc frame dau")
            continue
        H, W = probe.shape[:2]

        by_frame = collections.defaultdict(list)
        for line in open(ann):
            p = line.strip().split(",")
            if len(p) < 8:
                continue
            fi, _tid, x, y, w, h, score, cat = (int(float(v)) for v in p[:8])
            if score == 0:                 # vung bo qua, khong phai vat the
                n_skip_ignored += 1
                continue
            if cat not in MOT2YOLO:        # 0 = ignored, 11 = others
                n_skip_cat += 1
                continue
            cx, cy = (x + w / 2) / W, (y + h / 2) / H
            bw, bh = w / W, h / H
            if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                continue                   # box tran ra ngoai anh
            by_frame[fi].append(f"{MOT2YOLO[cat]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")

        for k, fname in enumerate(frames):
            if k % stride:
                continue
            fi = int(os.path.splitext(fname)[0])   # ten file la chi so frame, 1-based
            stem = f"{name}_{fi:07d}"
            link = os.path.join(img_out, stem + os.path.splitext(fname)[1])
            _link(os.path.join(seq, fname), link)
            with open(os.path.join(lab_out, stem + ".txt"), "w") as f:
                f.write("\n".join(by_frame.get(fi, [])))
            n_img += 1
            n_box += len(by_frame.get(fi, []))

        if (si + 1) % 10 == 0:
            print(f"    {si+1}/{len(seq_dirs)} sequence", flush=True)

    return n_img, n_box, n_skip_ignored, n_skip_cat, issues


splits = {}
if IS_MOT:
    print(f"Chuyen MOT -> YOLO (FRAME_STRIDE = {FRAME_STRIDE})\n")
    for split, want in (("train", "train"), ("val", "val")):
        root = find_mot_root(LOCAL, want)
        assert root, f"khong thay thu muc MOT cho split {split}"
        print(f"  [{split}] {os.path.basename(root)}")
        ni, nb, sk_ig, sk_cat, iss = convert_mot(root, split, FRAME_STRIDE)
        print(f"    {ni} anh, {nb} box  "
              f"(bo {sk_ig} vung ignored, {sk_cat} box lop 0/11)")
        for x in iss[:5]:
            print("    !!", x)
        splits[split] = os.path.join(YOLO_ROOT, split, "images")
    names, NC = VISDRONE_NAMES, len(VISDRONE_NAMES)
else:
    def find_split(root, s):
        for pat in (f"{s}/images", f"images/{s}", f"{s}", f"*{s}*/images"):
            for h in glob.glob(os.path.join(root, pat)):
                if os.path.isdir(h) and any(f.endswith(IMG_EXT) for f in os.listdir(h)):
                    return h
        return None
    splits = {s: find_split(LOCAL, s) for s in ("train", "val")}
    names, NC = VISDRONE_NAMES, len(VISDRONE_NAMES)
    for y in glob.glob(os.path.join(LOCAL, "**", "*.yaml"), recursive=True):
        d = yaml.safe_load(open(y))
        if isinstance(d, dict) and "names" in d:
            names = d["names"]
            if isinstance(names, dict):
                names = [names[k] for k in sorted(names)]
            NC = len(names)
            break

assert splits.get("train") and splits.get("val"), f"thieu split: {splits}"
print(f"\nnc = {NC}: {names}")

# ---- bon kiem tra, moi cai ung voi mot kieu hong kho truy -------------
problems, hist = [], collections.Counter()
for split, img_dir in splits.items():
    # Khong dung img_dir.replace("/images", "/labels"): tren he thong dung dau
    # phan cach khac thi replace khong khop, lab_dir van tro vao thu muc anh,
    # va vong kiem tra ben duoi dem duoc 0 box ma khong bao loi gi -- tuc la
    # bo kiem tra tu no hong mot cach im lang.
    lab_dir = os.path.join(os.path.dirname(img_dir), "labels")
    if not os.path.isdir(lab_dir):
        problems.append(f"{split}: khong thay {lab_dir}")
        continue

    imgs = [f for f in os.listdir(img_dir) if f.endswith(IMG_EXT)]
    missing = sum(1 for f in imgs if not os.path.exists(
        os.path.join(lab_dir, os.path.splitext(f)[0] + ".txt")))
    if missing:
        problems.append(f"{split}: {missing}/{len(imgs)} anh khong co file label")

    lfs = glob.glob(os.path.join(lab_dir, "*.txt"))
    bad_range = bad_cls = n_box = n_empty = 0
    for lf in random.Random(0).sample(lfs, min(500, len(lfs))):
        txt = open(lf).read().strip()
        if not txt:
            n_empty += 1
            continue
        for line in txt.split("\n"):
            p = line.split()
            if len(p) < 5:
                continue
            n_box += 1
            c = int(float(p[0]))
            hist[c] += 1
            if not (0 <= c < NC):
                bad_cls += 1
            if any(not (0.0 <= float(v) <= 1.0) for v in p[1:5]):
                bad_range += 1
    if bad_range:
        problems.append(f"{split}: {bad_range}/{n_box} box ngoai [0,1]")
    if bad_cls:
        problems.append(f"{split}: {bad_cls}/{n_box} box co class id ngoai [0,{NC})")
    print(f"  {split:5s} {len(imgs):6d} anh | {n_empty}/500 mau khong co vat the nao")

print("\nPhan bo lop (mau 500 file/split):")
mx = max(hist.values()) if hist else 1
for c in range(NC):
    print(f"  {c:2d} {names[c]:18s} {hist[c]:7d} {'#' * int(40 * hist[c] / mx)}")
    if hist[c] == 0:
        problems.append(f"lop {c} ({names[c]}) khong co box nao -> AP lop nay = -1")

DATA_YAML = os.path.join(os.path.dirname(LOCAL), "data.yaml")
yaml.safe_dump({"path": LOCAL, "train": splits["train"], "val": splits["val"],
                "nc": NC, "names": names},
               open(DATA_YAML, "w"), sort_keys=False, allow_unicode=True)

print("\n" + "=" * 62)
if problems:
    print("VAN DE - doc ky truoc khi train:")
    for p in problems:
        print("  !!", p)
else:
    print("Khong phat hien van de nao.")
print("=" * 62)
print(open(DATA_YAML).read())

## 8. Preflight — dựng model trước khi train

Ba tiếng train rồi mới biết model không dựng được là ba tiếng mất trắng. Cell
này dựng model, in số tham số, số anchor, và **tỉ lệ trọng số nạp được** —
con số cuối chính là bằng chứng cho caveat của p2, nên nó phải hiện ra chứ
không được im lặng.

In [ ]:
import warnings, io, contextlib, torch
warnings.filterwarnings("ignore")
from ultralytics import YOLO

buf = io.StringIO()
with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):
    _m = YOLO(SPEC["cfg"], verbose=False)
    _m.load(SPEC["weights"])
    _src = YOLO(SPEC["weights"]).model.state_dict()

# Dem truc tiep tren state_dict thay vi doc log: ultralytics in dong
# "Transferred x/y" qua LOGGER rieng, redirect_stdout khong bat duoc -- va mot
# cot im lang bao "n/a" chinh la cot khong ai kiem tra. Day dung la tieu chi
# ultralytics dung khi nap: trung ten VA trung shape.
_dst = _m.model.state_dict()
MATCHED = sum(1 for k, v in _dst.items()
              if k in _src and _src[k].shape == v.shape)
TRANSFER = f"{MATCHED}/{len(_dst)}"
TRANSFER_FRAC = MATCHED / len(_dst)

_head = _m.model.model[-1]
END2END = bool(getattr(_head, "end2end", False))
if END2END:
    _head.max_det = MAX_DET

_m.model.eval()
with torch.no_grad():
    _y = _m.model(torch.zeros(1, 3, IMGSZ, IMGSZ))
_out = _y[0] if isinstance(_y, (list, tuple)) else _y

N_PARAMS = sum(p.numel() for p in _m.model.parameters())
N_ANCHORS = None if END2END else int(_out.shape[-1])
# Preflight chay o nc=80 mac dinh cua yaml (model chua gap dataset), nen shape
# do duoc bay gio khong phai shape cuoi. Ghi ca hai de khong ai doc nham.
OUT_SHAPE_TRAINED = (1, MAX_DET, 6) if END2END else (1, 4 + NC, N_ANCHORS)

print(f"model            {MODEL}")
print(f"tham so          {N_PARAMS/1e6:.2f} M")
print(f"anchor           {N_ANCHORS if N_ANCHORS else '(end2end, khong dung anchor grid)'}")
print(f"NMS              {'KHONG can (end-to-end)' if END2END else 'can'}")
print(f"max_det          {MAX_DET if END2END else '(NMS quyet dinh)'}")
print(f"weights nap      {TRANSFER}  ({TRANSFER_FRAC*100:.0f}%)")
print(f"output sau train {OUT_SHAPE_TRAINED}")
print(f"  (preflight do duoc {tuple(_out.shape)} vi chay o nc=80 mac dinh cua yaml;")
print(f"   shape that duoc ghi lai tu file ONNX o cell 12)")

# Nguong 0.75 chu khong phai 0.9: model base thuong chi dat ~90% vi dau detect
# khong khop khi nc khac COCO -- do la binh thuong. Chi truong hop mat ca
# backbone/neck moi dang canh bao.
if TRANSFER_FRAC < 0.75:
    print(f"\n[luu y] Mot phan lon model khoi tao ngau nhien ({TRANSFER}).")
    print(f"        Da bu bang {SPEC['epochs']} epoch (cao hon {EPOCHS_BASE} cua cac model kia),")
    print(f"        nhung bang so sanh cuoi VAN phai ghi ro con so nay -- neu khong,")
    print(f"        nguoi doc se ket luan sai ve kien truc.")

del _m, _src, _dst
torch.cuda.empty_cache()

## 9. Đo bộ nhớ thật rồi tự chỉnh batch

`batch` ở cell 1 là ước lượng cho A100 40 GB. Cell này **đo thật** — vài bước
forward + backward với dữ liệu ngẫu nhiên rồi đọc đỉnh bộ nhớ — và tự **tăng**
nếu còn nhiều chỗ trống hoặc **giảm** nếu tràn.

Đây là cận dưới (chưa tính EMA và optimizer state, vốn nhỏ với model nano),
nhưng nó bắt OOM ở phút thứ hai thay vì giờ thứ hai.

In [ ]:
import torch, gc, warnings
warnings.filterwarnings("ignore")
from ultralytics import YOLO


def _all_tensors(o):
    """Dau train tra ve dict (yolo26 tra one2many/one2one), khong phai list.
    Gom het tensor lai roi tinh mot loss gia -- chi de do bo nho."""
    if torch.is_tensor(o):
        return [o]
    if isinstance(o, dict):
        o = list(o.values())
    if isinstance(o, (list, tuple)):
        out = []
        for x in o:
            out.extend(_all_tensors(x))
        return out
    return []


def peak_mem_gb(cfg, batch, imgsz=IMGSZ, steps=3):
    torch.cuda.empty_cache(); gc.collect()
    torch.cuda.reset_peak_memory_stats()
    m = YOLO(cfg, verbose=False).model.cuda().train()
    opt = torch.optim.SGD(m.parameters(), lr=1e-4)
    scaler = torch.amp.GradScaler("cuda")
    try:
        for _ in range(steps):
            x = torch.rand(batch, 3, imgsz, imgsz, device="cuda")
            with torch.amp.autocast("cuda"):
                ts = [t for t in _all_tensors(m(x)) if t.is_floating_point()]
                if not ts:
                    raise RuntimeError("khong lay duoc tensor nao tu dau ra")
                loss = sum((t.float() ** 2).mean() for t in ts)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
        peak = torch.cuda.max_memory_allocated() / 1e9
    except torch.cuda.OutOfMemoryError:
        peak = float("inf")
    finally:
        del m, opt
        torch.cuda.empty_cache(); gc.collect()
    return peak


BUDGET = VRAM_GB * 0.85     # chua 15% cho fragmentation va cudnn workspace
print(f"VRAM {VRAM_GB:.1f} GB -> ngan sach {BUDGET:.1f} GB\n")

batch = SPEC["batch"]
peak = peak_mem_gb(SPEC["cfg"], batch)
print(f"  batch {batch:4d} -> dinh {peak:5.1f} GB")

while peak > BUDGET and batch > 8:
    batch = max(8, batch // 2)
    peak = peak_mem_gb(SPEC["cfg"], batch)
    print(f"  ha xuong {batch:4d} -> dinh {peak:5.1f} GB")

while peak * 1.9 < BUDGET and batch < 512:
    trial = batch * 2
    p2 = peak_mem_gb(SPEC["cfg"], trial)
    if p2 > BUDGET:
        print(f"  thu {trial:4d} -> {p2:5.1f} GB, vuot ngan sach, giu {batch}")
        break
    batch, peak = trial, p2
    print(f"  tang len {batch:4d} -> dinh {peak:5.1f} GB")

SPEC["batch"] = batch
print(f"\n[chot] batch = {batch}  (dinh do duoc {peak:.1f} / {BUDGET:.1f} GB)")

## 10. Train

`project` trỏ **thẳng vào Drive** là có chủ ý: `/content` bị xoá sạch khi Colab
ngắt kết nối, nên checkpoint để ở đó thì mất trắng nhiều giờ. Để trên Drive thì
`last.pt` sống sót, và chạy lại đúng cell này là **tự resume**.

Dataset vẫn nằm ở `/content` (nhanh); chỉ checkpoint đi Drive. Đó là chỗ phân
chia đúng: ảnh đọc mỗi bước, checkpoint ghi mỗi epoch.

Augment: giữ photometric, **hạn chế geometric mạnh** — vật thể VisDrone rất nhỏ,
`scale`/`shear` lớn sẽ xoá sạch các box dưới 10 px.

In [ ]:
import os, glob, warnings, psutil
warnings.filterwarnings("ignore")
from ultralytics import YOLO

PROJECT = "/content/drive/MyDrive/skysentry/finetune_runs"
os.makedirs(PROJECT, exist_ok=True)

# --- Chon cache: don bay lon nhat ve toc do voi model nano ---------------
# Model nano tren A100 KHONG bi nghen o GPU. Tinh thu: yolov8n ~8.7 GFLOPs/anh
# o 640, train ~3x forward, tuc ~26 GFLOPs/anh. A100 chay fp16 thuc te vai chuc
# TFLOPS -> phan GPU cua mot epoch 6.5k anh chi vai giay. Cai an het thoi gian
# la giai nen JPEG tren CPU: mosaic doc 4 anh cho moi mau, batch 128 nghia la
# 512 lan decode anh 1920x1080 cho mot buoc, tren 12 vCPU cua Colab.
#
# cache="ram" giai nen mot lan roi giu anh da resize trong RAM -> vong lap
# khong con decode nua. Day thuong la khac biet 2-3x, lon hon moi thu khac
# trong cell nay cong lai.
_n_train = len([f for f in os.listdir(splits["train"]) if f.endswith(IMG_EXT)])
_ram_need = _n_train * IMGSZ * IMGSZ * 3 / 1e9        # can tren: anh vuong
_ram_free = psutil.virtual_memory().available / 1e9
_cpu = os.cpu_count() or 8

if _ram_need < _ram_free * 0.5:
    CACHE = "ram"
    _why = f"can ~{_ram_need:.1f} GB, con trong {_ram_free:.1f} GB"
elif _ram_need < 60:
    CACHE = "disk"
    _why = f"can ~{_ram_need:.1f} GB > 50% RAM trong ({_ram_free:.1f} GB) -> dung .npy tren dia"
else:
    CACHE = False
    _why = f"dataset qua lon ({_ram_need:.1f} GB)"

WORKERS = min(8, max(2, _cpu - 2))

print(f"anh train  : {_n_train}")
print(f"cache      : {CACHE}   ({_why})")
print(f"workers    : {WORKERS}  (thay {_cpu} vCPU)")
if CACHE == "ram":
    print("             -> epoch dau cham hon vi phai nap cache, cac epoch sau nhanh han")
print()

last = os.path.join(PROJECT, MODEL, "weights", "last.pt")
if os.path.exists(last):
    print(f"[resume] tim thay {last}\n")
    model = YOLO(last)
    resume = True
else:
    print(f"[moi] dung {SPEC['cfg']} + nap {SPEC['weights']}\n")
    model = YOLO(SPEC["cfg"], verbose=False)
    model.load(SPEC["weights"])
    resume = False

head = model.model.model[-1]
if getattr(head, "end2end", False):
    head.max_det = MAX_DET
    print(f"[e2e] max_det 300 -> {MAX_DET}\n")

results = model.train(
    data=DATA_YAML, epochs=SPEC["epochs"], imgsz=IMGSZ, batch=SPEC["batch"],
    device=0, workers=WORKERS, seed=SEED, deterministic=False,
    project=PROJECT, name=MODEL, exist_ok=True, resume=resume,
    patience=PATIENCE, amp=True, cache=CACHE, val=True, plots=True,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=0.0, translate=0.1, scale=0.5, shear=0.0, perspective=0.0,
    flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.0, close_mosaic=10,
)
print("\n[xong] train hoan tat")

# Da cham tran epoch hay da dung som? Voi p2 (chi nap 40% pretrained) day la
# cau hoi quan trong: cham tran nghia la mAP van con dang len khi het epoch,
# tuc con so cuoi cung la mot gioi han nhan tao, khong phai gioi han kien truc.
import pandas as pd
_csv = os.path.join(PROJECT, MODEL, "results.csv")
if os.path.exists(_csv):
    _d = pd.read_csv(_csv)
    _d.columns = [c.strip() for c in _d.columns]
    _n = len(_d)
    _col = next((c for c in _d.columns if "mAP50-95" in c), None)
    if _col:
        _best_ep = int(_d[_col].idxmax()) + 1
        print(f"  epoch da chay  : {_n}/{SPEC['epochs']}")
        print(f"  epoch tot nhat : {_best_ep}  (best.pt la cua epoch nay, khong phai epoch cuoi)")
        if _n < SPEC["epochs"]:
            print(f"\n  -> EARLY STOPPING da kich hoat: {PATIENCE} epoch lien tiep khong cai thien.")
            print(f"     Da bao hoa, con so nay dung la gioi han cua kien truc.")
        elif _best_ep > SPEC["epochs"] - PATIENCE:
            print(f"\n  [luu y] Cham tran {SPEC['epochs']} epoch, va epoch tot nhat nam o cuoi")
            print(f"          -> mAP CON DANG LEN khi het epoch. Con so nay la gioi han cua")
            print(f"          so epoch, KHONG phai cua kien truc. Muon ket luan cong bang thi")
            print(f"          tang EPOCHS_BASE o cell 1 roi chay lai cell nay -- no tu resume.")
        else:
            print(f"\n  -> Cham tran epoch nhung dinh nam o giua -> da bao hoa, ket luan dung.")

## 11. Validate theo giao thức VisDrone

`max_det=500`, không phải 300 mặc định. Ghi ra `summary.json` để cell gom kết
quả đọc được.

In [ ]:
import json, os
from ultralytics import YOLO

best = os.path.join(PROJECT, MODEL, "weights", "best.pt")
assert os.path.exists(best), f"khong thay {best}"

m = YOLO(best)
h = m.model.model[-1]
if getattr(h, "end2end", False):
    h.max_det = MAX_DET

res = m.val(data=DATA_YAML, imgsz=IMGSZ, batch=SPEC["batch"], device=0,
            max_det=MAX_DET, verbose=False)

summary = {
    "model": MODEL,
    "mAP50-95": float(res.box.map),
    "mAP50": float(res.box.map50),
    "mAP75": float(res.box.map75),
    "ap_per_class": {names[i]: float(v) for i, v in enumerate(res.box.maps)},
    "params_M": round(N_PARAMS / 1e6, 3),
    "anchors": N_ANCHORS,
    "end2end_no_nms": END2END,
    "max_det": MAX_DET,
    "weights_transferred": TRANSFER,
    "transfer_frac": round(TRANSFER_FRAC, 4),
    "epochs": SPEC["epochs"], "batch": SPEC["batch"], "imgsz": IMGSZ,
    "seed": SEED, "nc": NC,
    "output_shape_expected": list(OUT_SHAPE_TRAINED),
    "gpu": GPU_NAME,
    "ultralytics": ultralytics.__version__,
}
json.dump(summary, open(os.path.join(PROJECT, MODEL, "summary.json"), "w"),
          indent=2, ensure_ascii=False)

print(f"{MODEL}")
print(f"  mAP50-95 {summary['mAP50-95']:.4f}")
print(f"  mAP50    {summary['mAP50']:.4f}")
print(f"  mAP75    {summary['mAP75']:.4f}")
print("\nAP theo lop:")
for k, v in summary["ap_per_class"].items():
    print(f"  {k:18s} {v:.4f}")

## 12. Export ONNX cho pipeline QCS8550

Đúng thiết lập board yêu cầu, và vá sẵn lỗi đã gặp thật:

- `opset=13` — opset mới sinh op mà QNN đẩy ngược về CPU
- `dynamic=False`, shape tĩnh — bắt buộc để tạo QNN context binary
- **`sanitise_onnx()`** — Ultralytics + onnxslim để tensor output nằm cả trong
  `graph.output` lẫn `value_info`. ONNX Runtime bỏ qua, còn AI Hub **từ chối
  compile**: `Tensors {'output0'} occur in value_info but also in model IO`.
  Lỗi này đã chặn pipeline một lần rồi.
- `nms=False` chỉ truyền cho model cần NMS. Truyền cho model end2end là vô
  nghĩa và có thể làm export lỗi.

In [ ]:
import os, json, hashlib, shutil, warnings
warnings.filterwarnings("ignore")
from ultralytics import YOLO


def sha256(path, chunk=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()


def sanitise_onnx(path):
    """Bo tensor vua nam trong graph IO vua nam trong value_info."""
    import onnx
    mo = onnx.load(path)
    io_names = {t.name for t in mo.graph.input} | {t.name for t in mo.graph.output}
    dupes = [vi.name for vi in mo.graph.value_info if vi.name in io_names]
    if dupes:
        keep = [vi for vi in mo.graph.value_info if vi.name not in io_names]
        del mo.graph.value_info[:]
        mo.graph.value_info.extend(keep)
        onnx.save(mo, path)
    return dupes


OUT = os.path.join(PROJECT, MODEL)
m = YOLO(best)
h = m.model.model[-1]
if getattr(h, "end2end", False):
    h.max_det = MAX_DET

kw = dict(format="onnx", imgsz=IMGSZ, opset=13, dynamic=False,
          simplify=True, batch=1)
if not END2END:
    kw["nms"] = False
p = m.export(**kw)

onnx_path = os.path.join(OUT, f"{MODEL}.onnx")
shutil.move(str(p), onnx_path)
dupes = sanitise_onnx(onnx_path)

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
ishape = sess.get_inputs()[0].shape
oshape = sess.get_outputs()[0].shape

meta = {
    "model": MODEL,
    "onnx_sha256_16": sha256(onnx_path)[:16],
    "pt_sha256_16": sha256(best)[:16],
    "input_shape": ishape, "output_shape": oshape,
    "end2end_no_nms": END2END, "max_det": MAX_DET if END2END else None,
    "opset": 13, "imgsz": IMGSZ, "nc": NC,
    "value_info_dupes_removed": len(dupes),
    "pipeline_compatible": not END2END,
}
json.dump(meta, open(os.path.join(OUT, "export.json"), "w"), indent=2)

print(f"[ok] {onnx_path}")
print(f"     input  {ishape}")
print(f"     output {oshape}")
print(f"     sha256 {meta['onnx_sha256_16']}")
if dupes:
    print(f"     da va {len(dupes)} value_info trung (AI Hub se tu choi neu khong va)")

print("\n" + "=" * 64)
if END2END:
    print("KHONG tuong thich truc tiep voi 3-pipeline/detector.py")
    print(f"  output {oshape} = box da decode san [x1,y1,x2,y2,conf,cls]")
    print("  detector.py dang decode dang (1, 4+nc, A) -> cam vao se SAI THAM LANG.")
    print("  Can them nhanh: neu shape[-1]==6 thi bo NMS, chi loc theo conf,")
    print("  roi dua toa do ve he anh goc bang gain/pad nhu cu.")
else:
    print("Tuong thich voi 3-pipeline/detector.py")
    print(f"  output {oshape} - decode nhu cu, NMS chay tren Kryo va do rieng.")
print("=" * 64)

## 13. Gom kết quả từ mọi tab

Chạy cell này ở **bất kỳ tab nào sau khi các tab khác xong**. Nó đọc
`summary.json` của mọi model đã train xong trong cùng thư mục Drive.

Cột `weights nạp` phải có mặt: thiếu nó, bảng này sẽ bị đọc thành "kiến trúc p2
kém hơn", trong khi thực ra p2 chỉ xuất phát sau.

In [ ]:
import json, glob, os
import pandas as pd

rows = []
for sp in sorted(glob.glob(os.path.join(PROJECT, "*", "summary.json"))):
    s = json.load(open(sp))
    ep = os.path.join(os.path.dirname(sp), "export.json")
    e = json.load(open(ep)) if os.path.exists(ep) else {}
    rows.append({
        "model": s["model"],
        "mAP50-95": round(s["mAP50-95"], 4),
        "mAP50": round(s["mAP50"], 4),
        "mAP75": round(s["mAP75"], 4),
        "params(M)": s["params_M"],
        "anchors": s["anchors"] or "e2e",
        "NMS": "khong" if s["end2end_no_nms"] else "can",
        "max_det": s["max_det"],
        "epochs": s["epochs"],
        "batch": s["batch"],
        "weights nap": s["weights_transferred"],
        "onnx output": str(e.get("output_shape", "chua export")),
    })

if not rows:
    print("Chua co model nao xong.")
else:
    df = pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)
    display(df)
    out_csv = os.path.join(PROJECT, "comparison.csv")
    df.to_csv(out_csv, index=False)
    print(f"\n[ok] {out_csv}")
    print(f"[ok] {len(rows)}/4 model da xong: {', '.join(r['model'] for r in rows)}")

    print("\nDoc bang nay can nho:")
    print("  - Cot 'weights nap': model nao thap hon ~75% la xuat phat sau,")
    print("    khong phai kien truc kem hon.")
    print("  - Cot 'onnx output': dang (1,N,6) la box decode san, KHONG chay NMS;")
    print("    dang (1,4+nc,A) moi cam thang vao pipeline hien tai duoc.")

---

## Sau khi cả 4 model xong

**Bắt buộc trước khi dùng v26 trong pipeline.** `3-pipeline/detector.py` hiện
chỉ decode `(1, 4+nc, A)`. Hai model v26 trả `(1, N, 6)` đã decode sẵn. Cắm vào
mà không sửa thì **không có lỗi nào được báo** — chỉ là kết quả sai. Cần thêm
nhánh: `shape[-1] == 6` → tách `[x1, y1, x2, y2, conf, cls]`, bỏ NMS, lọc theo
`conf`, rồi đưa toạ độ về hệ ảnh gốc bằng `gain`/`pad` như cũ.

**Việc đáng làm nhất sau đó.** Bỏ NMS là bỏ **18 ms/frame** trên CPU — khoản
cắt lớn nhất còn lại trong frame budget (inference chỉ 47 ms). Nhưng phải kiểm
tra thật: đầu end-to-end dùng `topk`, và `topk` có thể bị QNN đẩy về CPU. Chạy
compile job trên AI Hub rồi đọc `n_ops_fallback` **trước khi** tin vào con số
này — nếu > 0 thì phần tiết kiệm sẽ bị trả lại ở chỗ khác.

**Mỗi dòng kết quả phải ghi kèm** `max_det` (500, không phải 300), tỉ lệ trọng
số nạp được, `imgsz`, `seed`, và sha256 của ONNX. Thiếu bất kỳ cái nào là hai
bảng không so được với nhau.